# Submission Inference

This notebook runs the deterministic preparation pipeline, rebuilds scenario-level features, trains the winning `hybrid_deep_sets_v2_family_only / raw` ensemble, validates the submission schema, and writes `outputs/predictions.csv`.

In [1]:
from __future__ import annotations

import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import sklearn
import torch

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print(f'root={ROOT}')
print(f'python={sys.version.split()[0]}')
print(f'platform={platform.platform()}')
print(f'numpy={np.__version__}')
print(f'pandas={pd.__version__}')
print(f'scipy={scipy.__version__}')
print(f'scikit-learn={sklearn.__version__}')
print(f'torch={torch.__version__}')
print(f'cuda_available={torch.cuda.is_available()}')


root=/Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid
python=3.11.0
platform=macOS-26.3-arm64-arm-64bit
numpy=2.4.4
pandas=3.0.2
scipy=1.17.1
scikit-learn=1.8.0
torch=2.11.0
cuda_available=False


In [2]:
from src.data.prepare_properties import run_preparation_pipeline
from src.features.build_scenario_features import run_feature_pipeline

prep_outputs = run_preparation_pipeline()
feature_outputs = run_feature_pipeline()

print('prepared_files=')
for name, path in prep_outputs.written_files.items():
    print(f'  {name}: {path}')
print(f"train_features_shape={feature_outputs.train_features.shape}")
print(f"test_features_shape={feature_outputs.test_features.shape}")


prepared_files=
  train_mixtures_normalized: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/mixtures_train_normalized.csv
  test_mixtures_normalized: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/mixtures_test_normalized.csv
  train_scenario_targets: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/processed/train_scenario_targets.csv
  component_properties_clean: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/component_properties_clean.csv
  component_property_catalog: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/component_property_catalog.csv
  component_properties_pivot_all: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/component_properties_pivot_all.csv
  component_properties_lookup_exact: /Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/data/interim/component_properties_lookup_exact.csv
  component_properties_lookup_typical: /Users/marwan/Downloads/Neftcode/neftekod-dot-

In [3]:
from src.models.train_deep_sets import (
    DeepSetsConfig,
    WINNING_TARGET_STRATEGY_NAME,
    WINNING_VARIANT_NAME,
    load_deep_sets_data,
    train_full_deep_sets_ensemble_and_predict,
)

ENSEMBLE_SEEDS = [0, 1, 2, 3, 4]
config = DeepSetsConfig()
prepared_data = load_deep_sets_data()
predictions = train_full_deep_sets_ensemble_and_predict(
    prepared_data=prepared_data,
    variant_name=WINNING_VARIANT_NAME,
    target_strategy_name=WINNING_TARGET_STRATEGY_NAME,
    seeds=ENSEMBLE_SEEDS,
    config=config,
)
predictions.head()


,scenario_id,"Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %","Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm"
0,test_1,-37.248405,49.060555
1,test_10,-3.131000,59.956055
2,test_11,25.284990,70.756500
3,test_12,-13.406197,32.521709
4,test_13,-6.785894,29.988403


In [4]:
expected_columns = [
    'scenario_id',
    'Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %',
    'Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm',
]

assert predictions.columns.tolist() == expected_columns, predictions.columns.tolist()
assert len(predictions) == 40, len(predictions)
assert predictions['scenario_id'].nunique() == len(predictions)
assert not predictions.duplicated(subset=['scenario_id']).any()
assert not predictions.isnull().any().any()

output_dir = ROOT / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)
prediction_path = output_dir / 'predictions.csv'
predictions.to_csv(prediction_path, index=False, encoding='utf-8')

print(f'prediction_path={prediction_path}')
print(f'rows={len(predictions)} cols={len(predictions.columns)}')
print(predictions.head().to_string(index=False))


prediction_path=/Users/marwan/Downloads/Neftcode/neftekod-dot-hybrid/outputs/predictions.csv
rows=40 cols=3
scenario_id  Delta Kin. Viscosity KV100 - relative | - Daimler Oxidation Test (DOT), %  Oxidation EOT | DIN 51453 Daimler Oxidation Test (DOT), A/cm
     test_1                                                                 -37.248405                                                     49.060555
    test_10                                                                  -3.131000                                                     59.956055
    test_11                                                                  25.284990                                                     70.756500
    test_12                                                                 -13.406197                                                     32.521709
    test_13                                                                  -6.785894                                                     29.98840